### Lab Assignment: Commercial Data Analysis

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: February 15, 2026

---

### INSTRUCTIONS  
In this assignment, you will work with a dataset containing information about businesses.  
Each record is a business location.  Follow the steps below, writing and running the code in blocks, and displaying the solutions.  

The path to the dataset is in a file named `find_dataset_on_rivanna.txt` in Module 3. 

Each question part is worth 1 POINT, for a total of 15 POINTS.

Hint: reaching deeper fields in json hierarchy can be done like this:  

`df.select('address.street_number')`

---

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
        .appName("comm") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 17:25:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# note that read.json can read a zipped JSON directly

**1. (1 PT) Read in the dataset and show the number of records**

In [4]:
path = "/standard/ds7200-apt4c/large_datasets/part-00000-a159c41a-bc58-4476-9b78-c437667f9c2b-c000.json.gz"

# schema = StructType([
#     StructField("date", TimestampType(), True)])
data = spark.read.json(path)

**2. (1 PT) Show the first 3 records**

In [6]:
data.show(3)

+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|             address|       business_tags|               hours|              id|menu|             reviews|                urls|             webpage|
+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|{Woodburn, {45.15...|                NULL|                NULL|000023995a540868|NULL|                  []|{woodburn.k12.or....|{Educational Tech...|
|{Hialeah, {25.884...|{[], [{has_atm, Y...|{NULL, 1900, NULL...|0000821a1394916e|NULL|                NULL|{NULL, [yelp.com]...|                NULL|
|{Rochester, {43.1...|{[], [{accepts_cr...|{NULL, 1700, NULL...|000136e65d50c3b7|NULL|[{New (to me) qui...|{usps.com, [yelp....|{Welcome | USPS G...|
+--------------------+--------------------+--------------------+----------------+----+--------------

**3. (1 PT) Show the first 5 street addresses which are not null**  

In [7]:
data.select("address").filter(data.address.isNotNull()).show(5)

+--------------------+
|             address|
+--------------------+
|{Woodburn, {45.15...|
|{Hialeah, {25.884...|
|{Rochester, {43.1...|
|{West Palm Beach,...|
|{Eufaula, {35.283...|
+--------------------+
only showing top 5 rows


**4. (1 PT) Location**  

Count the number of records where the city is New York

In [8]:
data.where(data.address.city=="New York").count()

26/06/12 17:25:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

1953

**5. (1 PT) Hours**  

Count the number of records where closing time on Tuesday is 8pm

In [9]:
data.where(F.col("hours.tuesday_close") == "2000").count()

3154

**6. (1 PT) Location and Hours**  

For the records where the city is New York, aggregate by Tuesday closing time, showing a column called `count` with the number of records for each Tuesday closing time. Sort by the `count` column in descending order. Here is an example of results output:

+-------------+-----+  
|tuesday_close|count|  
+-------------+-----+  
|         1800| 2001|  
|         1700| 1800|  
|         2000| 1550|  

In [10]:
data.where(F.col("address.city")=="New York").groupBy("hours.tuesday_close").agg(F.count("hours.tuesday_close")).sort("hours.tuesday_close").show(50)

[Stage 10:>                                                         (0 + 1) / 1]

+-------------+--------------------------+
|tuesday_close|count(hours.tuesday_close)|
+-------------+--------------------------+
|         NULL|                         0|
|         0000|                        15|
|         0100|                         5|
|         0200|                         1|
|         0300|                         1|
|         0359|                         1|
|         0400|                         5|
|         0530|                         1|
|         0830|                         1|
|         0900|                         1|
|         1000|                         1|
|         1100|                         1|
|         1200|                         2|
|         1230|                         2|
|         1400|                         2|
|         1430|                         2|
|         1500|                         4|
|         1600|                        10|
|         1630|                         4|
|         1645|                         1|
|         1

**7. (1 PT) Price Range**  

Price range is quoted in number of dollar signs.  Count the number of records with price range greater than or equal to three.

In [11]:
data.where(data.menu.price_range>=3).select("menu.price_range").count()

115

**8. (1 PT) Missing Webpage URL**  

Count the number of records that are missing the webpage url.

In [12]:
data.where(data.webpage.url.isNull()).count()

79813

**9. (1 PT) Webpage URLs**  

Register the dataframe as a temp table.  
Next, use Spark SQL to select only the webpage title column, filtering on rows where the webpage url (accessed under `webpage.url`) is *Target.com*. 

Show only one resulting row and don't truncate the output.

In [13]:
data.createOrReplaceTempView("businesses")
spark.sql("SELECT webpage.title FROM businesses WHERE webpage.url=='Target.com'").show(1, truncate=False)

+-------------------------------+
|title                          |
+-------------------------------+
|Target : Expect More. Pay Less.|
+-------------------------------+
only showing top 1 row


**10. (1 PT) Analysis on Ratings**  

The reviews contains information such as the number of stars for each review (the *rating*).  
The ratings are stored in an array (`reviews.stars`) for each business location (you should check for yourself). Return the top five most common rating arrays.  For example, an array might look like: 
[5, 5]



In [14]:
data.groupBy("reviews.stars").count().sort(F.desc("count")).show(5)

[Stage 20:>                                                         (0 + 1) / 1]

+------+-----+
| stars|count|
+------+-----+
|  NULL|74679|
|    []|42419|
|   [5]| 4258|
|[NULL]| 3067|
|[5, 5]| 1610|
+------+-----+
only showing top 5 rows


**11. More work with Ratings**  

For this question, you will filter out null ratings and then compute the average rating for each business location (using the field: `id`).


a) (1 PT) Create a new dataframe retaining two fields: `id`, `reviews.stars`


In [15]:
reviews_df = data.select("id", "reviews.stars")
reviews_df.show(5)

+----------------+------+
|              id| stars|
+----------------+------+
|000023995a540868|    []|
|0000821a1394916e|  NULL|
|000136e65d50c3b7|[4, 4]|
|00014329a70b9869|  NULL|
|00031c0a83f00657|  NULL|
+----------------+------+
only showing top 5 rows


b) (1 PT) Create a row for each rating  
hint: use the `withColumn()` and `explode()` functions  
you will need to import the `explode()` function by issuing:

`from pyspark.sql.functions import explode`


In [16]:
# already imported functions as F
reviews_df = reviews_df.withColumn("rating", F.explode('stars'))
reviews_df.show(5)

+----------------+--------------------+------+
|              id|               stars|rating|
+----------------+--------------------+------+
|000136e65d50c3b7|              [4, 4]|     4|
|000136e65d50c3b7|              [4, 4]|     4|
|0003b7589a4e12a0|                 [5]|     5|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
+----------------+--------------------+------+
only showing top 5 rows


c) (1 PT) Return a count of the number of ratings in this dataframe

In [17]:
reviews_df.count()

600082

d) (1 PT) Drop rows where the rating is null, and return a count of the number of non-null ratings

In [18]:
reviews_df = reviews_df.where(reviews_df.rating.isNotNull())
reviews_df.count()

538241

e) (1 PT) Compute the average rating, grouped by `id`. After the average is computed, sort by `id` in ascending order and show the top 10 records.  
 
hint:   
this can all be done in one line using the `agg()` function  
this `id` should be at the top: 000136e65d50c3b7

In [24]:
reviews_df.groupBy(reviews_df.id).agg(F.mean("rating")).sort(F.asc("id")).show(10)

[Stage 34:>                                                         (0 + 1) / 1]

+----------------+------------------+
|              id|       avg(rating)|
+----------------+------------------+
|000136e65d50c3b7|               4.0|
|0003b7589a4e12a0|               5.0|
|00059519f0dba1b4|3.3333333333333335|
|000a1df4c8e0ecd2|               4.6|
|000c7b7a30623083|               5.0|
|000c9ffc8b89af03|               3.0|
|000de20baa847ecc|1.6666666666666667|
|001064359d9f162f|               5.0|
|0010c9f495d87dd7|               3.0|
|0017774db5e6400a| 4.333333333333333|
+----------------+------------------+
only showing top 10 rows
